# RAG (Retrieval-Augmented Generation) & Tool Agent

**RAG** adalah arsitektur yang menggabungkan dua komponen utama:
1. **Retriever** — mencari dokumen/chunk relevan dari knowledge base berdasarkan query
2. **Generator** — LLM yang menghasilkan jawaban berdasarkan dokumen yang ditemukan

Tanpa RAG, LLM hanya mengandalkan pengetahuan dari training data (yang bisa outdated atau tidak spesifik). Dengan RAG, LLM bisa menjawab berdasarkan dokumen terbaru, privat, atau domain-spesifik.

**Tool Agent** mengembangkan konsep ini lebih jauh — LLM tidak hanya membaca dokumen, tapi bisa **memanggil tool/fungsi** (search web, query database, kalkulasi, dll.) secara dinamis untuk menyelesaikan tugas.

Notebook ini mencakup:
- RAG pipeline dengan **LlamaIndex** dan **LangChain**
- Tool Agent dengan **LlamaIndex** dan **LangChain**
- Komponen-komponen kunci: Document Loader, Text Splitter, Embedding, VectorStore, Retriever, Chain/Agent

### 1. Install & Import Library
Kedua framework membutuhkan package tambahan. LlamaIndex dan LangChain memiliki ekosistem modular — kita install komponen yang diperlukan saja. Untuk LLM, kita gunakan OpenAI sebagai contoh (bisa diganti dengan HuggingFace, Ollama, dll.).

In [ ]:
# Install (jalankan sekali jika belum terinstall)
# !pip install llama-index llama-index-llms-openai llama-index-embeddings-openai
# !pip install llama-index-vector-stores-chroma llama-index-readers-file
# !pip install langchain langchain-openai langchain-community langchain-chroma
# !pip install chromadb sentence-transformers faiss-cpu pypdf
# !pip install openai tiktoken

import os
import warnings
warnings.filterwarnings('ignore')

# === Konfigurasi API Key ===
# Ganti dengan API key Anda atau gunakan environment variable
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# Cek library yang tersedia
libs = {}
for lib, import_name in [
    ('llama_index',     'llama_index.core'),
    ('langchain',       'langchain'),
    ('chromadb',        'chromadb'),
    ('sentence_transformers', 'sentence_transformers'),
    ('faiss',           'faiss'),
    ('pypdf',           'pypdf'),
]:
    try:
        __import__(import_name)
        libs[lib] = True
        print(f"  ✓ {lib}")
    except ImportError:
        libs[lib] = False
        print(f"  ✗ {lib} — belum terinstall")

print("\nCatatan: Semua cell di notebook ini menampilkan kode yang bisa dijalankan")
print("setelah API key dikonfigurasi dan dependencies terinstall.")

---
## BAGIAN A: KONSEP DASAR RAG

### 2. Konsep: Alur Pipeline RAG
Sebelum masuk ke kode, penting memahami setiap komponen dalam pipeline RAG dan apa fungsinya. Alur lengkap RAG:

```
INDEXING (sekali saja):
  Dokumen → [DocumentLoader] → [TextSplitter] → chunks
           → [EmbeddingModel] → vectors
           → [VectorStore] (simpan ke database)

QUERYING (setiap query):
  User Query → [EmbeddingModel] → query vector
             → [Retriever] → top-k chunks relevan
             → [Prompt Template] → context + query
             → [LLM] → jawaban akhir
```

Komponen yang perlu dipilih/dikonfigurasi:
| Komponen | Pilihan Populer |
|----------|----------------|
| **LLM** | OpenAI GPT, HuggingFace, Ollama (lokal), Groq |
| **Embedding Model** | OpenAI `text-embedding-ada-002`, `all-MiniLM-L6-v2`, `BAAI/bge-small-en` |
| **Vector Store** | ChromaDB (lokal), FAISS (in-memory), Pinecone (cloud), Weaviate |
| **Text Splitter** | RecursiveCharacterTextSplitter, SentenceSplitter, SemanticSplitter |
| **Retriever** | Dense (semantic), Sparse (BM25), Hybrid |

In [ ]:
# Simulasi dokumen untuk seluruh notebook
# (digunakan saat library tersedia dan API key dikonfigurasi)

SAMPLE_DOCS = [
    """
    Transformer Architecture Overview.
    The Transformer model, introduced in 'Attention Is All You Need' (Vaswani et al., 2017),
    revolutionized natural language processing. It relies entirely on self-attention mechanisms,
    dispensing with recurrence and convolutions. The encoder processes input tokens in parallel
    using multi-head attention, while the decoder generates output tokens autoregressively.
    Key components include: Multi-Head Attention, Position-wise Feed-Forward Networks,
    Positional Encoding, and Layer Normalization.
    """,
    """
    BERT: Bidirectional Encoder Representations from Transformers.
    BERT (Devlin et al., 2018) uses a Transformer encoder to create bidirectional representations.
    It is pre-trained using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
    BERT-base has 12 layers, 768 hidden dimensions, 12 attention heads, and 110M parameters.
    BERT-large has 24 layers, 1024 hidden dimensions, 16 attention heads, and 340M parameters.
    BERT achieved state-of-the-art results on 11 NLP tasks when released.
    """,
    """
    GPT (Generative Pre-trained Transformer).
    GPT models use the Transformer decoder architecture for causal language modeling.
    GPT-2 has 1.5B parameters and generates coherent long-form text.
    GPT-3 has 175B parameters and introduced few-shot learning capabilities.
    GPT-4 is a multimodal model with improved reasoning and instruction following.
    GPT models are trained on large corpora using next-token prediction as the objective.
    """,
    """
    Fine-tuning Large Language Models.
    Fine-tuning adapts a pre-trained LLM to specific tasks or domains.
    Full fine-tuning updates all model parameters, requiring significant compute.
    Parameter-Efficient Fine-Tuning (PEFT) methods like LoRA and QLoRA update only a small
    subset of parameters, making fine-tuning accessible on consumer hardware.
    LoRA adds low-rank adapter matrices to attention layers, typically adding <1% extra parameters.
    Instruction fine-tuning (like Alpaca, FLAN) improves instruction-following capability.
    """,
    """
    RAG: Retrieval-Augmented Generation.
    RAG combines a retrieval system with a generative LLM to reduce hallucination.
    Documents are chunked, embedded, and stored in a vector database.
    At query time, relevant chunks are retrieved and provided as context to the LLM.
    RAG is particularly useful for domain-specific knowledge, reducing the need for full fine-tuning.
    Key metrics for RAG evaluation include: faithfulness, answer relevancy, context recall, context precision.
    """,
]

print(f"Dokumen sample siap: {len(SAMPLE_DOCS)} dokumen")
for i, doc in enumerate(SAMPLE_DOCS):
    title = doc.strip().split('\n')[0].strip()
    print(f"  [{i}] {title}")

---
## BAGIAN B: RAG dengan LLAMAINDEX

### 3. [LlamaIndex] Setup LLM dan Embedding Model
LlamaIndex adalah framework yang sangat modular — semua komponen (LLM, embedding, vector store) dapat diganti dengan mudah. `Settings` adalah objek global untuk konfigurasi default. Selain OpenAI, bisa digunakan model lokal via Ollama atau HuggingFace.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# === Opsi 1: OpenAI (membutuhkan API key) ===
# llm = OpenAI(model='gpt-4o-mini', temperature=0.1)
# embed_model = OpenAIEmbedding(model='text-embedding-3-small')

# === Opsi 2: HuggingFace Embedding (lokal, gratis) ===
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

# === Opsi 3: Ollama (LLM lokal, gratis) ===
# from llama_index.llms.ollama import Ollama
# llm = Ollama(model='llama3', request_timeout=60.0)

# === Konfigurasi Global Settings ===
# Settings.llm         = llm
# Settings.embed_model = embed_model
# Settings.chunk_size  = 512
# Settings.chunk_overlap = 50

print("=== LlamaIndex: Opsi LLM dan Embedding ===")
config_info = """
OPSI LLM:
  OpenAI GPT  : OpenAI(model='gpt-4o-mini')          ← paling mudah, berbayar
  Ollama      : Ollama(model='llama3')                ← lokal, gratis, butuh RAM besar  
  HuggingFace : HuggingFaceLLM(model_name='...')     ← fleksibel, gratis
  Groq        : Groq(model='llama3-70b-8192')        ← cepat, ada free tier

OPSI EMBEDDING:
  OpenAI      : OpenAIEmbedding(model='text-embedding-3-small')  ← akurat, berbayar
  BAAI/bge    : HuggingFaceEmbedding('BAAI/bge-small-en-v1.5')  ← gratis, bagus
  MiniLM      : HuggingFaceEmbedding('all-MiniLM-L6-v2')        ← ringan, gratis

OPSI VECTOR STORE:
  In-memory   : SimpleVectorStore (default LlamaIndex)           ← tidak persisten
  ChromaDB    : ChromaVectorStore(chroma_collection=...)         ← persisten, lokal
  Pinecone    : PineconeVectorStore(pinecone_index=...)          ← cloud, skalabel
"""
print(config_info)

### 4. [LlamaIndex] Document Loading dan Indexing
LlamaIndex mendukung berbagai format dokumen: PDF, DOCX, HTML, CSV, Markdown, bahkan URL. Dokumen dipecah menjadi **nodes** (chunks) menggunakan `NodeParser`, kemudian setiap node di-embed dan disimpan ke vector store. Proses ini hanya dilakukan sekali dan hasilnya bisa dipersist ke disk.

In [ ]:
from llama_index.core import (
    VectorStoreIndex, Document, StorageContext, load_index_from_storage
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

# === Cara 1: Buat Document dari string Python ===
documents = [
    Document(
        text=doc.strip(),
        metadata={'source': f'doc_{i}', 'topic': 'LLM'}
    )
    for i, doc in enumerate(SAMPLE_DOCS)
]
print(f"[Doc dari string] Jumlah dokumen: {len(documents)}")
print(f"  Contoh metadata: {documents[0].metadata}")
print(f"  Panjang teks doc[0]: {len(documents[0].text)} karakter")

# === Cara 2: Load dari PDF ===
print("\n[Load PDF]")
pdf_code = '''
from llama_index.readers.file import PDFReader

loader = PDFReader()
pdf_docs = loader.load_data(file=Path('receipt_info.pdf'))
print(f"PDF: {len(pdf_docs)} halaman dimuat")
'''
print(pdf_code)

# === Cara 3: Load dari direktori (semua file sekaligus) ===
print("[Load direktori]")
dir_code = '''
from llama_index.core import SimpleDirectoryReader

# Otomatis deteksi format: PDF, DOCX, TXT, MD, HTML, CSV
reader = SimpleDirectoryReader(
    input_dir='./documents',
    recursive=True,               # termasuk subdirektori
    required_exts=['.pdf', '.txt', '.md']  # filter ekstensi
)
docs = reader.load_data()
print(f"{len(docs)} dokumen dimuat dari direktori")
'''
print(dir_code)

# === Text Splitter: SentenceSplitter ===
splitter = SentenceSplitter(
    chunk_size=256,          # maksimal karakter per chunk
    chunk_overlap=32,        # overlap antar chunk untuk konteks
    paragraph_separator='\n\n'
)

nodes = splitter.get_nodes_from_documents(documents)
print(f"\n[Node Parser] {len(documents)} dokumen → {len(nodes)} nodes")
for i, node in enumerate(nodes[:3]):
    print(f"  Node {i}: '{node.text[:80].strip()}...' (len={len(node.text)})")

### 5. [LlamaIndex] VectorStoreIndex dan Query Engine
`VectorStoreIndex` adalah jantung dari RAG di LlamaIndex. Ia melakukan embedding semua nodes dan menyimpannya ke vector store. `as_query_engine()` langsung membuat pipeline RAG lengkap: query → retrieve → LLM → response. Bisa juga dikustomisasi via `RetrieverQueryEngine`.

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import get_response_synthesizer

# ============================================================
# PENTING: Ganti komentar di bawah dengan API key aktif
# sebelum menjalankan cell ini.
# ============================================================

# === Build Index (proses embedding semua nodes) ===
# index = VectorStoreIndex(nodes)  # in-memory

# === Simpan index ke disk agar tidak perlu embed ulang ===
# index.storage_context.persist(persist_dir='./storage_llamaindex')

# === Load index dari disk ===
# storage_context = StorageContext.from_defaults(persist_dir='./storage_llamaindex')
# index = load_index_from_storage(storage_context)

# === Query Engine Dasar ===
# query_engine = index.as_query_engine(
#     similarity_top_k=3,          # ambil 3 chunk paling relevan
#     response_mode='compact',     # 'compact', 'refine', 'tree_summarize'
#     streaming=False
# )
# response = query_engine.query('What is BERT and how many parameters does it have?')
# print(response)
# print('\nSource nodes (chunk yang digunakan):')
# for src in response.source_nodes:
#     print(f'  Score: {src.score:.4f} | {src.text[:100]}...')

# === Query Engine Kustom (lebih kontrol) ===
print("=== LlamaIndex: RetrieverQueryEngine Kustom ===")
custom_qe_code = '''
# Retriever kustom
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5,
)

# Post-processor: filter chunk dengan score terlalu rendah
postprocessor = SimilarityPostprocessor(similarity_cutoff=0.7)

# Response synthesizer
synthesizer = get_response_synthesizer(
    response_mode='tree_summarize',  # rangkum dari beberapa chunks
    use_async=False,
)

# Rakit query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    node_postprocessors=[postprocessor]
)

response = query_engine.query('Explain the difference between BERT and GPT.')
print(response)
'''
print(custom_qe_code)

print("\n=== Response Mode Options ===")
print("""
  'refine'          : Proses chunk satu per satu, refine jawaban berulang kali
  'compact'         : Gabungkan chunk ke prompt, query LLM sekali (default, hemat token)
  'tree_summarize'  : Bangun tree dari chunk, rangkum dari bawah ke atas
  'simple_summarize': Truncate semua chunk ke satu prompt
  'no_text'         : Kembalikan source nodes saja tanpa generate teks
""")

### 6. [LlamaIndex] ChromaDB sebagai Persistent Vector Store
In-memory vector store hilang saat kernel restart. **ChromaDB** adalah vector database lokal yang persisten — data disimpan ke disk dan bisa diload kembali. Cocok untuk development dan deployment ringan tanpa membutuhkan cloud service.

In [ ]:
# from llama_index.vector_stores.chroma import ChromaVectorStore
# import chromadb

print("=== LlamaIndex + ChromaDB (Persistent Vector Store) ===")
chroma_code = '''
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

# ── INDEXING (sekali saja) ──────────────────────────────────
# Inisialisasi ChromaDB client (persisten ke disk)
chroma_client = chromadb.PersistentClient(path='./chroma_db')
chroma_collection = chroma_client.get_or_create_collection('llm_docs')

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build dan simpan index ke ChromaDB
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    show_progress=True
)
print("Index disimpan ke ChromaDB.")

# ── QUERYING (load dari ChromaDB yang sudah ada) ────────────
chroma_client2   = chromadb.PersistentClient(path='./chroma_db')
chroma_coll2     = chroma_client2.get_collection('llm_docs')
vector_store2    = ChromaVectorStore(chroma_collection=chroma_coll2)
storage_ctx2     = StorageContext.from_defaults(vector_store=vector_store2)

index_loaded = VectorStoreIndex.from_vector_store(vector_store2)
query_engine = index_loaded.as_query_engine(similarity_top_k=3)

response = query_engine.query('How many parameters does BERT-large have?')
print(response)
'''
print(chroma_code)

### 7. [LlamaIndex] Chat Engine (RAG dengan Memory/History)
**Chat Engine** mengembangkan Query Engine dengan menambahkan **conversational memory** — konteks percakapan sebelumnya dipertahankan. Ini penting untuk chatbot berbasis dokumen di mana pengguna bisa merujuk ke pertanyaan sebelumnya (`'Tell me more about that'`).

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

print("=== LlamaIndex: Chat Engine dengan Memory ===")
chat_code = '''
from llama_index.core.memory import ChatMemoryBuffer

# Chat memory: simpan N token terakhir dari percakapan
memory = ChatMemoryBuffer.from_defaults(token_limit=3000)

chat_engine = index.as_chat_engine(
    chat_mode='condense_plus_context',  # mode terbaik untuk RAG+chat
    memory=memory,
    verbose=True,
    system_prompt=(
        "You are an expert AI assistant specializing in Large Language Models. "
        "Answer questions based on the provided context. "
        "If information is not in the context, say so clearly."
    )
)

# Simulasi percakapan multi-turn
queries = [
    "What is BERT?",
    "How many layers does it have?",          # merujuk ke BERT dari atas
    "What about GPT? How is it different?",
    "Which one is better for text generation?",
]

for q in queries:
    print(f"\nUser: {q}")
    response = chat_engine.chat(q)
    print(f"Assistant: {response}")

# Reset memory untuk percakapan baru
chat_engine.reset()
'''
print(chat_code)

print("\n=== Chat Mode Options ===")
print("""
  'best'                  : Otomatis pilih mode terbaik
  'condense_question'     : Kondensasi history + query jadi satu query baru
  'context'               : Selalu retrieve context, abaikan history
  'condense_plus_context' : Gabungan — kondensasi + retrieve context (TERBAIK)
  'simple'                : Chat biasa tanpa retrieval
  'react'                 : ReAct agent mode (dengan reasoning steps)
""")

---
## BAGIAN C: RAG dengan LANGCHAIN

### 8. [LangChain] Setup LLM, Embedding, dan Vector Store
LangChain menggunakan paradigma **chain** dan **LCEL (LangChain Expression Language)** dengan operator `|` (pipe). Arsitektur ini sangat eksplisit — setiap langkah pipeline terlihat jelas dan mudah dikustomisasi atau diganti komponen-per-komponen.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document as LCDocument
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("=== LangChain: Setup Komponen ===")
setup_code = '''
# ── LLM ──────────────────────────────────────────────────────
# OpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# HuggingFace (via API)
from langchain_huggingface import HuggingFaceEndpoint
llm = HuggingFaceEndpoint(repo_id="mistralai/Mistral-7B-Instruct-v0.2", temperature=0.1)

# Ollama (lokal)
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3", temperature=0)

# ── EMBEDDING ────────────────────────────────────────────────
# OpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# HuggingFace (gratis, lokal)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# ── VECTOR STORE ─────────────────────────────────────────────
# ChromaDB (persisten)
vectorstore = Chroma(
    collection_name="llm_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain"
)

# FAISS (in-memory, cepat)
vectorstore = FAISS.from_texts(texts, embeddings)
vectorstore.save_local("faiss_index")   # simpan ke disk
vectorstore = FAISS.load_local("faiss_index", embeddings)  # load kembali
'''
print(setup_code)

### 9. [LangChain] Document Loader, Text Splitter, dan Indexing
LangChain memiliki **200+ document loaders** untuk berbagai sumber: PDF, web, database, Notion, YouTube, GitHub, dll. **RecursiveCharacterTextSplitter** adalah text splitter yang paling sering digunakan — ia memecah teks secara hierarkis (coba split di `\n\n`, lalu `\n`, lalu spasi, dst.) untuk menjaga paragraf tetap utuh.

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)
from langchain_core.documents import Document as LCDocument

# === Buat dokumen LangChain dari string ===
lc_docs = [
    LCDocument(
        page_content=doc.strip(),
        metadata={'source': f'doc_{i}', 'topic': 'LLM'}
    )
    for i, doc in enumerate(SAMPLE_DOCS)
]
print(f"Dokumen LangChain: {len(lc_docs)}")

# === RecursiveCharacterTextSplitter ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', ''],  # prioritas pemisah
    add_start_index=True,   # tambahkan posisi awal chunk ke metadata
)

chunks = text_splitter.split_documents(lc_docs)
print(f"\n[RecursiveCharacterSplitter] {len(lc_docs)} dokumen → {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:4]):
    print(f"  Chunk {i}: len={len(chunk.page_content)} | meta={chunk.metadata}")
    print(f"    '{chunk.page_content[:80].strip()}...'")

# === Berbagai Loader (kode referensi) ===
print("\n=== Daftar Document Loaders LangChain ===")
loaders_ref = '''
# PDF
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("receipt_info.pdf")
pages  = loader.load()   # satu Document per halaman

# Web scraping URL tunggal
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://arxiv.org/abs/1706.03762")
docs   = loader.load()

# Direktori file teks
from langchain_community.document_loaders import DirectoryLoader, TextLoader
loader = DirectoryLoader('./docs', glob='**/*.txt', loader_cls=TextLoader)
docs   = loader.load()

# CSV
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path='data.csv', source_column='text')
docs   = loader.load()

# YouTube transcript
from langchain_community.document_loaders import YoutubeLoader
loader = YoutubeLoader.from_youtube_url("https://youtube.com/watch?v=...")
docs   = loader.load()
'''
print(loaders_ref)

### 10. [LangChain] RAG Chain dengan LCEL (Pipe Operator `|`)
**LCEL (LangChain Expression Language)** menggunakan operator `|` untuk merangkai komponen menjadi pipeline yang efisien, mendukung streaming, async, dan batching secara otomatis. Pipeline RAG klasik: `retriever | prompt | llm | output_parser`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

print("=== LangChain: RAG Chain dengan LCEL ===")

# ── Prompt Template RAG ──────────────────────────────────────
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use ONLY the following retrieved context to answer the question.
If the answer is not in the context, say "I don't have information about that."
Be concise and accurate.

Context:
{context}

Question: {question}

Answer:"""
)

rag_chain_code = '''
# ── Setup ────────────────────────────────────────────────────
# Build vectorstore dan retriever dari chunks yang sudah dibuat
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(
    search_type='similarity',   # 'similarity', 'mmr', 'similarity_score_threshold'
    search_kwargs={'k': 3}      # ambil 3 chunk paling relevan
)

# ── Fungsi helper format dokumen ─────────────────────────────
def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

# ── RAG Chain (LCEL) ─────────────────────────────────────────
rag_chain = (
    RunnableParallel(
        context  = retriever | format_docs,  # retrieve → format
        question = RunnablePassthrough()     # query pass-through
    )
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# ── Invoke ────────────────────────────────────────────────────
answer = rag_chain.invoke("What is BERT and how many parameters does BERT-base have?")
print("Jawaban:", answer)

# ── Streaming ────────────────────────────────────────────────
for chunk in rag_chain.stream("Explain LoRA fine-tuning."):
    print(chunk, end="", flush=True)

# ── Batch (beberapa query sekaligus) ──────────────────────────
answers = rag_chain.batch([
    "What is RAG?",
    "How does GPT differ from BERT?",
    "What is fine-tuning?"
])
'''
print(rag_chain_code)

# Tampilkan prompt template untuk referensi
print("\n=== Prompt Template yang Digunakan ===")
print(RAG_PROMPT.format(context='[retrieved chunks here]', question='[user question]'))

### 11. [LangChain] Conversational RAG dengan Chat History
RAG dengan memory membutuhkan dua chain: (1) **Contextualize chain** — reformulasi query berdasarkan history percakapan menjadi pertanyaan yang berdiri sendiri, (2) **QA chain** — jawab pertanyaan yang sudah dikontekstualisasi menggunakan retrieved docs. Ini adalah pattern **Conversational RAG** standar di LangChain.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("=== LangChain: Conversational RAG ===")
conv_rag_code = '''
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.messages import HumanMessage, AIMessage

# ── Step 1: Contextualize query berdasar chat history ─────────
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and latest user question, "
     "reformulate it as a standalone question. "
     "Do NOT answer it, just reformulate."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

# ── Step 2: QA Chain ──────────────────────────────────────────
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI assistant. Answer based ONLY on this context:\\n\\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# ── Step 3: Gabungkan menjadi RAG chain dengan history ────────
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# ── Simulasi percakapan ───────────────────────────────────────
chat_history = []
questions = [
    "What is BERT?",
    "How many layers does it have?",
    "Is GPT similar to it?"
]

for question in questions:
    response = rag_chain.invoke({
        "input": question,
        "chat_history": chat_history
    })
    answer = response["answer"]
    
    print(f"User: {question}")
    print(f"AI  : {answer}\\n")
    
    # Update history
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer)
    ])
'''
print(conv_rag_code)

---
## BAGIAN D: TOOL AGENT

### 12. Konsep Tool Agent
**Agent** adalah LLM yang dapat **memutuskan sendiri** tool mana yang perlu dipanggil, dengan argumen apa, dan kapan berhenti. Berbeda dengan RAG yang alurnya linier (query → retrieve → generate), Agent bersifat **iteratif** dan adaptif.

```
User Query
    ↓
LLM (Reasoning: tool mana yang diperlukan?)
    ↓
Tool Call (misal: search('LoRA paper'))
    ↓
Tool Result (dikembalikan ke LLM)
    ↓
LLM (apakah sudah cukup? butuh tool lain?)
    ↓  (loop jika perlu)
Final Answer
```

Framework reasoning populer:
| Framework | Cara Kerja |
|-----------|------------|
| **ReAct** | Reason + Act — LLM berfikir (Thought) lalu bertindak (Action) bergantian |
| **OpenAI Function Calling** | LLM langsung output JSON tool call yang terstruktur |
| **Tool Calling** | Versi generik function calling untuk model non-OpenAI |
| **Plan & Execute** | Buat rencana dulu, eksekusi langkah per langkah |

In [ ]:
# Definisikan tool-tool kustom yang akan digunakan di section berikutnya
import json
import math
from datetime import datetime

# Simulasi tool functions (tanpa API key)
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Input: string like '2 + 2' or 'sqrt(16)'."""
    try:
        allowed = {'sqrt': math.sqrt, 'pow': math.pow, 'pi': math.pi, 'e': math.e,
                   'abs': abs, 'round': round, 'log': math.log}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return f"Result: {result}"
    except Exception as ex:
        return f"Error: {ex}"

def get_current_time(timezone: str = 'UTC') -> str:
    """Get the current date and time. Input: timezone string."""
    return f"Current time ({timezone}): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def search_knowledge_base(query: str) -> str:
    """Search the AI/LLM knowledge base for information."""
    query_lower = query.lower()
    for doc in SAMPLE_DOCS:
        if any(word in doc.lower() for word in query_lower.split()[:3]):
            return doc.strip()[:400] + '...'
    return "No relevant information found in knowledge base."

def get_model_info(model_name: str) -> str:
    """Get information about a specific AI model."""
    models_db = {
        'bert':      'BERT-base: 12 layers, 768 dim, 110M params. BERT-large: 24 layers, 340M params.',
        'gpt-2':     'GPT-2: 1.5B parameters, Transformer decoder, trained on WebText.',
        'gpt-3':     'GPT-3: 175B parameters, few-shot learning, 96 layers.',
        'gpt-4':     'GPT-4: Multimodal, improved reasoning, ~1.8T parameters (estimated).',
        'llama':     'LLaMA 2: 7B, 13B, 34B, 70B variants. Open-source from Meta.',
        'mistral':   'Mistral 7B: Sliding Window Attention, outperforms Llama 2 13B.',
    }
    key = model_name.lower().replace('-', '').replace('_', '')
    for k, v in models_db.items():
        if k in key or key in k:
            return v
    return f"Model '{model_name}' not found in database."

# Test tools
print("=== Test Tool Functions ===")
print(f"calculator('2 ** 10')    : {calculator('2 ** 10')}")
print(f"calculator('sqrt(144)')  : {calculator('sqrt(144)')}")
print(f"get_current_time()       : {get_current_time()}")
print(f"get_model_info('bert')   : {get_model_info('bert')}")
print(f"search_kb('BERT layers') : {search_knowledge_base('BERT layers')[:100]}...")

### 13. [LlamaIndex] FunctionCallingAgent dengan Tools
LlamaIndex menyediakan `FunctionCallingAgent` yang menggunakan kemampuan function calling native dari model OpenAI. Setiap Python function dibungkus menjadi `FunctionTool` — LlamaIndex secara otomatis mengekstrak nama, deskripsi, dan parameter dari docstring dan type hints.

In [ ]:
from llama_index.core.tools import FunctionTool, QueryEngineTool

print("=== LlamaIndex: FunctionCallingAgent ===")

# Bungkus fungsi Python menjadi FunctionTool
calc_tool = FunctionTool.from_defaults(
    fn=calculator,
    name='calculator',
    description='Useful for evaluating mathematical expressions and calculations. '
                'Input should be a valid math expression string.'
)

time_tool = FunctionTool.from_defaults(
    fn=get_current_time,
    name='get_current_time',
    description='Returns the current date and time. '
                'Use when the user asks about current time or date.'
)

model_info_tool = FunctionTool.from_defaults(
    fn=get_model_info,
    name='get_model_info',
    description='Retrieves technical information about AI/LLM models such as BERT, GPT, LLaMA, etc.'
)

kb_tool = FunctionTool.from_defaults(
    fn=search_knowledge_base,
    name='search_knowledge_base',
    description='Searches the AI/LLM knowledge base for information about topics '
                'like transformers, fine-tuning, RAG, attention mechanisms, etc.'
)

print("Tools yang terdaftar:")
for tool in [calc_tool, time_tool, model_info_tool, kb_tool]:
    print(f"  - {tool.metadata.name}: {tool.metadata.description[:70]}...")

# ── Buat Agent ────────────────────────────────────────────────
agent_code = '''
from llama_index.core.agent import FunctionCallingAgent

tools = [calc_tool, time_tool, model_info_tool, kb_tool]

agent = FunctionCallingAgent.from_tools(
    tools=tools,
    llm=llm,
    verbose=True,    # tampilkan reasoning steps
    system_prompt=(
        "You are a helpful AI research assistant specializing in Large Language Models. "
        "Use the available tools to find accurate information before answering. "
        "Always cite which tool you used."
    )
)

# Test queries yang membutuhkan tool berbeda
queries = [
    "What time is it right now?",
    "How many parameters does BERT-large have?",
    "If BERT-base has 110M parameters and BERT-large has 340M, what is the ratio?",
    "Explain how RAG works and how is it different from fine-tuning?",
]

for q in queries:
    print(f"\\n{'='*60}")
    print(f"Query: {q}")
    print("="*60)
    response = agent.chat(q)
    print(f"Answer: {response}")
'''
print(agent_code)

### 14. [LlamaIndex] QueryEngineTool — RAG sebagai Tool Agent
`QueryEngineTool` mengubah RAG Query Engine menjadi salah satu tool yang bisa digunakan agent. Ini memungkinkan **multi-document RAG agent** — agent dapat memilih dari beberapa knowledge base yang berbeda untuk menjawab query yang kompleks.

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

print("=== LlamaIndex: Multi-Source RAG Agent ===")
multi_rag_code = '''
from llama_index.core.agent import FunctionCallingAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# ── Bangun beberapa index (knowledge base berbeda) ─────────────
# Index 1: Dokumen teknis AI
ai_docs    = [doc for doc in all_documents if doc.metadata["type"] == "technical"]
ai_index   = VectorStoreIndex.from_documents(ai_docs)
ai_engine  = ai_index.as_query_engine(similarity_top_k=3)

# Index 2: Paper arxiv
paper_docs  = [doc for doc in all_documents if doc.metadata["type"] == "paper"]
paper_index = VectorStoreIndex.from_documents(paper_docs)
paper_engine = paper_index.as_query_engine(similarity_top_k=3)

# ── Bungkus jadi tool ─────────────────────────────────────────
ai_tool = QueryEngineTool(
    query_engine=ai_engine,
    metadata=ToolMetadata(
        name="ai_technical_docs",
        description="Contains technical documentation about AI architectures, "
                    "BERT, GPT, Transformers, fine-tuning methods, and RAG."
    )
)

paper_tool = QueryEngineTool(
    query_engine=paper_engine,
    metadata=ToolMetadata(
        name="research_papers",
        description="Contains AI research papers including original transformer paper, "
                    "BERT, GPT series, LoRA, and evaluation benchmarks."
    )
)

# ── Multi-source Agent ────────────────────────────────────────
tools = [ai_tool, paper_tool, calc_tool, time_tool]

agent = FunctionCallingAgent.from_tools(
    tools=tools,
    llm=llm,
    verbose=True,
)

# Agent akan OTOMATIS memilih tool yang tepat
response = agent.chat(
    "Compare BERT and GPT architectures, then calculate: "
    "if BERT has 12 attention heads and GPT-3 has 96, "
    "how many times more heads does GPT-3 have?"
)
print(response)
'''
print(multi_rag_code)

### 15. [LangChain] Tool Agent dengan `@tool` Decorator dan ReAct
LangChain menyediakan dekorator `@tool` yang sangat mudah digunakan — cukup tambahkan ke fungsi Python biasa dan docstring otomatis menjadi deskripsi tool. **ReAct agent** menggunakan pola Thought-Action-Observation berulang kali hingga mendapatkan jawaban final.

In [ ]:
from langchain.tools import tool
from langchain_core.tools import BaseTool

# === Definisikan tools dengan @tool decorator ===
@tool
def lc_calculator(expression: str) -> str:
    """Evaluate a mathematical expression. 
    Use for calculations, percentages, arithmetic operations.
    Input: a valid math expression string like '2**10' or '(110 + 340) / 2'
    """
    return calculator(expression)

@tool
def lc_get_current_time(timezone: str = 'UTC') -> str:
    """Get the current date and time in a specified timezone.
    Use when user asks about current time, today's date, or what day it is.
    Input: timezone string (e.g., 'UTC', 'WIB', 'EST')
    """
    return get_current_time(timezone)

@tool  
def lc_model_info(model_name: str) -> str:
    """Look up technical information about an AI/LLM model.
    Use when user asks about model parameters, architecture, or specifications.
    Input: model name like 'BERT', 'GPT-3', 'LLaMA', 'Mistral'
    """
    return get_model_info(model_name)

@tool
def lc_search_kb(query: str) -> str:
    """Search the AI knowledge base for information about LLMs, transformers, fine-tuning, RAG.
    Use for conceptual questions about AI topics.
    Input: search query string
    """
    return search_knowledge_base(query)

lc_tools = [lc_calculator, lc_get_current_time, lc_model_info, lc_search_kb]

print("LangChain Tools terdaftar:")
for t in lc_tools:
    print(f"  - {t.name}: {t.description[:70].strip()}...")

# ── Kode Agent ────────────────────────────────────────────────
print("\n=== LangChain: ReAct Agent ===")
react_code = '''
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# Pull ReAct prompt dari LangChain hub
react_prompt = hub.pull("hwchase17/react")

# Buat agent
react_agent = create_react_agent(llm, lc_tools, react_prompt)

# AgentExecutor mengelola loop tool-calling
agent_executor = AgentExecutor(
    agent=react_agent,
    tools=lc_tools,
    verbose=True,           # tampilkan Thought-Action-Observation
    max_iterations=5,       # batas maksimum langkah
    handle_parsing_errors=True,
    return_intermediate_steps=True  # kembalikan semua langkah
)

# Jalankan
result = agent_executor.invoke({
    "input": "What is BERT? How many parameters does BERT-large have, "
             "and what is that number squared?"
})
print("\\nFinal answer:", result["output"])
print("\\nLangkah-langkah:")
for step in result["intermediate_steps"]:
    action, observation = step
    print(f"  Action: {action.tool}({action.tool_input})")
    print(f"  Observation: {str(observation)[:100]}...")
'''
print(react_code)

### 16. [LangChain] OpenAI Tools Agent (Function Calling) — Lebih Andal dari ReAct
`create_openai_tools_agent` menggunakan **function calling native** dari OpenAI — model langsung mengoutput JSON terstruktur untuk tool calls, jauh lebih andal daripada ReAct yang mengandalkan parsing teks bebas. Ini adalah rekomendasi untuk production.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# System prompt untuk agent
AGENT_SYSTEM_PROMPT = """You are a helpful AI research assistant specializing in Large Language Models.
You have access to several tools:
- Use calculator for math operations
- Use get_current_time for time/date queries  
- Use model_info for AI model specifications
- Use search_kb for conceptual AI/ML questions

Always use the appropriate tool rather than relying on your training data alone.
Be accurate, concise, and cite your sources."""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),  # tool call history
])

print("=== LangChain: OpenAI Tools Agent ===")
tools_agent_code = '''
from langchain.agents import create_openai_tools_agent, AgentExecutor

# Buat agent dengan function calling
tools_agent = create_openai_tools_agent(llm, lc_tools, agent_prompt)

executor = AgentExecutor(
    agent=tools_agent,
    tools=lc_tools,
    verbose=True,
    max_iterations=6,
    return_intermediate_steps=True
)

# Complex multi-step query
complex_query = (
    "1. What is RAG and how does it work? "
    "2. BERT-large has 340M parameters and GPT-3 has 175B. "
    "   What percentage of GPT-3's size is BERT-large? "
    "3. Also, what is today's date?"
)

result = executor.invoke({"input": complex_query})
print(result["output"])

# ── Dengan Chat History (stateful agent) ─────────────────────
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}  # simple in-memory store

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

agent_with_history = RunnableWithMessageHistory(
    executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Session 1
result1 = agent_with_history.invoke(
    {"input": "What is BERT?"},
    config={"configurable": {"session_id": "user_001"}}
)

# Session 1 (lanjut — agent ingat BERT dari sebelumnya)
result2 = agent_with_history.invoke(
    {"input": "How many parameters does it have?"},
    config={"configurable": {"session_id": "user_001"}}
)
'''
print(tools_agent_code)

### 17. [LangChain] RAG + Agent: Retriever Tool dalam Agent
Menggabungkan RAG dengan Agent — retriever dibungkus menjadi `create_retriever_tool` sehingga agent bisa **memilih kapan** perlu mencari dokumen. Cocok untuk chatbot yang bisa menjawab dari knowledge base, melakukan kalkulasi, dan berinteraksi dengan API external sekaligus.

In [ ]:
from langchain.tools.retriever import create_retriever_tool

print("=== LangChain: RAG + Agent (Retriever as Tool) ===")
rag_agent_code = '''
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import create_openai_tools_agent, AgentExecutor

# ── Build Vectorstore dan Retriever ──────────────────────────
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

# ── Bungkus Retriever sebagai Tool ───────────────────────────
retriever_tool = create_retriever_tool(
    retriever,
    name="llm_knowledge_base",
    description=(
        "Search the knowledge base for information about Large Language Models, "
        "transformers, BERT, GPT, fine-tuning, LoRA, RAG, and AI research. "
        "Use this tool for any technical AI questions."
    )
)

# ── Gabungkan semua tools ─────────────────────────────────────
all_tools = [
    retriever_tool,        # RAG
    lc_calculator,         # Math
    lc_get_current_time,   # Time
    lc_model_info,         # Model specs
]

# ── Build Agent ───────────────────────────────────────────────
agent = create_openai_tools_agent(llm, all_tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True, max_iterations=7)

# Test: query yang membutuhkan kombinasi tools
result = executor.invoke({"input": (
    "From the knowledge base, find information about GPT and BERT. "
    "Then tell me: what year is it now, and how many years have passed "
    "since the original Transformer paper (2017)?"
)})
print(result["output"])
'''
print(rag_agent_code)

### 18. Evaluasi RAG: Metrik dan Framework
Evaluasi RAG tidak bisa hanya mengandalkan inspeksi manual. Ada metrik kuantitatif yang perlu diukur. **RAGAS** adalah framework evaluasi RAG yang paling populer — ia menggunakan LLM sebagai hakim untuk menilai kualitas retrieval dan generation.

In [ ]:
print("=== Metrik Evaluasi RAG ===")

metrics_explanation = """
RETRIEVAL METRICS:
┌─────────────────────┬─────────────────────────────────────────────────────┐
│ Metrik              │ Definisi                                             │
├─────────────────────┼─────────────────────────────────────────────────────┤
│ Context Recall      │ Seberapa banyak informasi ground truth               │
│                     │ ditemukan di retrieved context?                     │
│ Context Precision   │ Seberapa relevan retrieved context dengan pertanyaan?│
│ Context Relevancy   │ Apakah retrieved chunks benar-benar berguna?        │
└─────────────────────┴─────────────────────────────────────────────────────┘

GENERATION METRICS:
┌─────────────────────┬─────────────────────────────────────────────────────┐
│ Faithfulness        │ Apakah jawaban konsisten dengan retrieved context?  │
│                     │ (deteksi halusinasi) — skor ideal: 1.0              │
│ Answer Relevancy    │ Seberapa relevan jawaban dengan pertanyaan asal?    │
│ Answer Correctness  │ Apakah jawaban faktual benar? (butuh ground truth)  │
└─────────────────────┴─────────────────────────────────────────────────────┘
"""
print(metrics_explanation)

print("=== Evaluasi dengan RAGAS ===")
ragas_code = '''
# pip install ragas
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

# Siapkan dataset evaluasi: question, answer, contexts, ground_truth
eval_data = {
    "question"    : ["What is BERT?", "How many parameters does GPT-3 have?"],
    "answer"      : [
        "BERT is a bidirectional transformer model pre-trained with MLM and NSP.",
        "GPT-3 has 175 billion parameters."
    ],
    "contexts"    : [
        ["BERT uses bidirectional encoder representations..."],  # retrieved chunks
        ["GPT-3 has 175B parameters..."],
    ],
    "ground_truth": [
        "BERT stands for Bidirectional Encoder Representations from Transformers.",
        "GPT-3 has 175 billion parameters."
    ]
}

dataset = Dataset.from_dict(eval_data)

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision]
)

print(result.to_pandas())
# Output: tabel dengan skor 0-1 untuk setiap metrik per pertanyaan
'''
print(ragas_code)

print("\n=== Ringkasan Perbandingan LlamaIndex vs LangChain ===")
comparison = """
┌─────────────────┬────────────────────────────┬────────────────────────────┐
│                 │ LlamaIndex                 │ LangChain                  │
├─────────────────┼────────────────────────────┼────────────────────────────┤
│ Fokus utama     │ Data ingestion & RAG        │ Chain & Agent orchestration │
│ Kemudahan RAG   │ ★★★★★ (sangat mudah)        │ ★★★★☆ (eksplisit)          │
│ Fleksibilitas   │ ★★★★☆                       │ ★★★★★ (sangat fleksibel)   │
│ Ekosistem Tool  │ ★★★☆☆                       │ ★★★★★ (200+ integrations)  │
│ LCEL/Pipe       │ Tidak ada                   │ ✓ (composable chains)      │
│ Streaming       │ ✓ (built-in)                │ ✓ (LCEL native)            │
│ Vector Stores   │ 20+                         │ 50+                        │
│ Best for        │ Dokumen-heavy RAG            │ Complex agent workflows    │
└─────────────────┴────────────────────────────┴────────────────────────────┘

REKOMENDASI:
  → RAG sederhana (Q&A atas dokumen): LlamaIndex
  → Multi-step agent + banyak tools: LangChain
  → Production chatbot kompleks: Kombinasi keduanya
"""
print(comparison)